In [1]:
!pip install groq python-dotenv --quiet

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 138.3/138.3 kB 3.9 MB/s eta 0:00:00


In [2]:
from dotenv import load_dotenv
import os
import getpass

load_dotenv()

if "GROQ_API_KEY" not in os.environ:
    os.environ["GROQ_API_KEY"] = getpass.getpass("Enter GROQ API KEY: ")

Enter GROQ API KEY: ··········


In [10]:
from groq import Groq

client = Groq(api_key=os.environ["GROQ_API_KEY"])

BASE_MODEL = "llama-3.1-8b-instant"

In [11]:
MODEL_CONFIG = {
    "technical": {
        "system_prompt": """
You are a Technical Support Expert.
Provide precise, structured, code-focused debugging help.
Explain errors clearly and suggest fixes.
"""
    },

    "billing": {
        "system_prompt": """
You are a Billing Support Expert.
Respond empathetically.
Explain payment issues, refunds, and subscription policies clearly.
"""
    },

    "general": {
        "system_prompt": """
You are a General Customer Support Assistant.
Handle casual conversation and general inquiries politely.
"""
    },

    "tool": {
        "system_prompt": """
You are a Tool Routing Expert.
If financial or real-time data is requested,
use external tools instead of guessing.
"""
    }
}

In [12]:
def route_prompt(user_input):

    routing_prompt = f"""
Classify the following user query into one category:

technical
billing
general
tool

Return ONLY the category name.

Query: {user_input}
"""

    response = client.chat.completions.create(
        model=BASE_MODEL,
        temperature=0,
        messages=[
            {"role": "user", "content": routing_prompt}
        ]
    )

    category = response.choices[0].message.content.strip().lower()
    return category

In [13]:
def get_bitcoin_price():
    return "Current Bitcoin price is approximately $65,000 (mock data)."

In [14]:
def call_expert(category, user_input):

    system_prompt = MODEL_CONFIG[category]["system_prompt"]

    response = client.chat.completions.create(
        model=BASE_MODEL,
        temperature=0.7,
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": user_input}
        ]
    )

    return response.choices[0].message.content

In [15]:
def process_request(user_input):

    category = route_prompt(user_input)
    print("Routed To:", category)

    if category == "tool":
        return get_bitcoin_price()

    if category not in MODEL_CONFIG:
        category = "general"

    return call_expert(category, user_input)

In [16]:
queries = [
    "My python code gives IndexError in list access",
    "I was charged twice for my subscription",
    "Hello how are you",
    "What is the current price of Bitcoin?"
]

for q in queries:
    print("\nUser:", q)
    print("Response:", process_request(q))


User: My python code gives IndexError in list access
Routed To: technical
Response: **IndexError in List Access**

An `IndexError` in Python occurs when you try to access an element at an index that does not exist in a list. This can happen when:

* You are trying to access an element at the end of the list, which does not exist.
* You are trying to access an element at a negative index, which is not allowed in Python lists.
* You are assuming the list has a certain length, but it is shorter than expected.

**Example Code**
```python
my_list = [1, 2, 3]
print(my_list[5])  # IndexError
```
In this example, `my_list` only has three elements, but we are trying to access the element at index 5, which does not exist.

**Error Message**
```python
File "example.py", line 3, in <module>
    print(my_list[5])
IndexError: list index out of range
```
**Fixing the Error**

To fix the `IndexError`, you need to ensure that you are accessing an index that exists in the list. Here are a few possible 